# Agent 概述

Agent 是一种由大模型驱动的任务执行方式：模型不只是生成一段文本，而是根据目标自主判断下一步要做什么，必要时调用工具、读取结果、继续推理，直到给出最终答案。

简单理解：
- **LLM**：回答问题
- **Tool Calling**：模型决定要调用哪个工具，但工具执行通常由外层代码完成
- **Agent**：模型决定行动，框架负责执行工具，并把工具结果继续交给模型，形成多轮执行闭环


![Agent 完整工作流](resources/agent-complete-workflow.png)


## 1. Agent 解决什么问题

普通 LLM 的能力边界主要来自两个方面：
- 只能基于输入上下文和模型已有知识回答
- 不能自己执行外部动作，例如搜索、查数据库、发请求、计算、写文件

Agent 通过给模型配置工具，让模型可以在需要时执行外部动作。例如：
- 查询实时信息
- 调用业务系统 API
- 检索私有知识库
- 执行计算或代码
- 按步骤完成一个复杂任务


## 2. Agent、Chain、Workflow 的区别

| 类型 | 核心特点 | 适合场景 |
|------|----------|----------|
| LLM | 单次输入，单次输出 | 问答、改写、摘要、分类 |
| Chain | 开发者预先编排步骤 | 固定流程，例如提取 -> 检索 -> 总结 |
| Workflow | 明确的业务状态和分支 | 审批流、工单流、确定性业务流程 |
| Agent | 模型动态决定下一步 | 开放式任务、工具选择不固定、步骤数量不确定 |

如果流程非常稳定，优先使用 Chain 或 Workflow；如果用户问题变化很大，需要模型自己判断工具和步骤，再考虑 Agent。

## 3. Agent 的核心组成

一个 Agent 通常包含以下部分：

| 组成 | 作用 |
|------|------|
| Model | 决定下一步动作，并生成最终回答 |
| Tools | Agent 可调用的外部能力，例如搜索、数据库、计算函数 |
| System Prompt | 约束 Agent 的角色、边界、工具使用规则和输出格式 |
| Memory | 保存会话历史、用户偏好、长期知识等上下文 |
| Runtime / Executor | 负责执行工具调用，并把结果返回给模型 |

其中最关键的是 **工具描述**。模型不会直接理解 Python 函数本身，而是根据工具名、参数 schema 和 docstring 判断什么时候使用工具。

## 4. Agent 的执行循环

Agent 的典型执行流程如下：

1. 用户提出任务
2. 模型分析当前上下文，判断是否需要调用工具
3. 如果需要工具，模型生成工具名和参数
4. Agent Executor 执行工具
5. 工具结果作为新上下文返回给模型
6. 模型继续判断：继续调用工具，或生成最终答案

这个循环可以让 Agent 完成多步任务，但也意味着需要控制最大轮数、工具权限和异常处理。

In [ ]:
# 基础依赖
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv()

## 5. 最小 Agent 示例

下面定义两个工具：
- 查询订单状态
- 查询物流信息

Agent 会根据用户问题，自己决定是否调用工具以及调用哪个工具。

In [ ]:
@tool
def get_order_status(order_id: str) -> str:
    """根据订单号查询订单状态。"""
    mock_orders = {
        "A1001": "已付款，等待仓库打包",
        "A1002": "已发货",
        "A1003": "已签收",
    }
    return mock_orders.get(order_id, "未查询到该订单")


@tool
def get_shipping_info(order_id: str) -> str:
    """根据订单号查询物流信息。"""
    mock_shipping = {
        "A1002": "顺丰速运 SF123456789，预计明天送达",
        "A1003": "顺丰速运 SF987654321，已于昨天签收",
    }
    return mock_shipping.get(order_id, "暂未查询到物流信息")

In [ ]:
model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek"
)

system_prompt = """
你是一个订单客服助手。
回答订单问题时，必须优先调用工具查询真实状态。
如果工具没有查到信息，明确告诉用户无法查询到，不要编造订单状态。
"""

agent = create_agent(
    model=model,
    tools=[get_order_status, get_shipping_info],
    system_prompt=system_prompt,
)

In [ ]:
response = agent.invoke({
    "messages": [HumanMessage(content="帮我查一下订单 A1002 现在到哪了？")]
})

rprint(response)

## 6. Agent 中 Prompt 的作用

Agent 的 System Prompt 不只是角色设定，更重要的是约束行为边界：

- 什么时候必须调用工具
- 什么时候不能调用工具
- 工具查不到结果时如何回答
- 是否允许猜测、补全、编造
- 输出格式是否固定
- 遇到敏感操作时是否需要确认

示例：

```text
你是一个订单客服助手。
回答订单问题时必须调用订单查询工具。
如果工具没有返回结果，不要编造。
涉及退款、取消订单、修改地址时，必须先向用户确认。
```


## 7. Tools 设计原则

工具设计会直接影响 Agent 的稳定性。

好的工具应该具备：
- 工具名清晰，例如 `get_order_status` 比 `query` 更好
- docstring 说明准确，告诉模型什么时候使用
- 参数尽量结构化，不要只给一个模糊的 `text`
- 返回值简洁，不要塞入大量无关内容
- 对异常和空结果有明确返回

不推荐把一个工具做得过于宽泛。例如 `business_tool(action, payload)` 会让模型更难判断参数，也更难调试。

## 8. Agent 与 Memory

Agent 默认只知道当前输入。如果希望它记住上下文，需要引入 Memory。

常见记忆类型：
- **短期记忆**：当前会话内的多轮消息
- **长期记忆**：跨会话保存的用户偏好、历史记录、业务信息
- **摘要记忆**：当消息过长时，把历史内容压缩成摘要

Agent 使用 Memory 后，可以处理连续对话，例如：

```text
用户：帮我查 A1002
助手：订单已发货...
用户：那它大概什么时候到？
```

第二个问题里的“它”需要依赖上一轮上下文才能正确理解。

## 9. Agent 与结构化输出

很多业务场景不只是需要自然语言回答，还需要稳定的数据结构。例如：
- 工单分类结果
- 风险等级
- 是否需要人工介入
- 下一步动作

这类场景可以把 Agent 和结构化输出结合起来：

```python
class TicketDecision(BaseModel):
    category: str
    priority: str
    need_human: bool
    reply: str
```

这样可以减少自由文本带来的解析成本，方便后续系统继续处理。

## 10. 常见 Agent 类型

| 类型 | 说明 |
|------|------|
| 工具型 Agent | 根据问题选择并调用工具 |
| 搜索型 Agent | 先搜索外部信息，再总结回答 |
| RAG Agent | 在知识库检索、追问和总结之间动态切换 |
| 数据分析 Agent | 调用 SQL、Python、图表工具完成分析 |
| 多 Agent | 多个 Agent 分工协作，例如规划、执行、审查 |
| Supervisor Agent | 一个上级 Agent 负责把任务分配给其他 Agent |

学习时建议先掌握单 Agent + Tools，再扩展到 Memory、RAG 和多 Agent。

## 11. 什么时候不适合使用 Agent

Agent 并不是所有场景的首选。

以下情况更适合固定流程：
- 每一步都很确定，不需要模型动态决策
- 对延迟和成本非常敏感
- 工具调用必须严格按顺序执行
- 错误动作成本很高，例如转账、删除数据、发送正式通知
- 业务流程要求可审计、可预测、可回放

在生产系统里，常见做法是：**关键流程用 Workflow 控制，开放式判断交给 Agent，敏感动作加人工确认**。

## 12. Agent 开发中的常见问题

| 问题 | 原因 | 处理方式 |
|------|------|----------|
| 不调用工具 | Prompt 约束不清，工具描述不清 | 明确什么时候必须调用工具 |
| 调错工具 | 工具职责重叠 | 拆分工具边界，优化名称和描述 |
| 参数错误 | 参数 schema 太模糊 | 使用 Pydantic 描述参数 |
| 编造结果 | 工具空结果没有约束 | Prompt 中禁止编造，工具返回明确空状态 |
| 循环调用 | 目标不清或结果不足 | 设置最大迭代次数，增加停止条件 |
| 成本过高 | 多轮调用太多 | 限制工具数量，优化流程，缓存结果 |


## 13. 小结

Agent 的本质是：**让大模型在工具和上下文的支持下，动态决定下一步行动**。

掌握 Agent 需要关注四个问题：
- 模型能调用哪些工具
- 工具描述是否清晰
- Prompt 是否定义了行为边界
- 执行过程是否可控、可观测、可恢复

后续学习可以围绕以下顺序展开：
1. Agent 调用工具
2. Agent 使用搜索
3. Agent 使用记忆
4. Agent 结合 RAG
5. 多 Agent 协作
